In [0]:
import subprocess
import sys
import re
import time
import os
import atexit
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pyspark import SparkConf
from pyspark import SparkContext
from pyspark import SQLContext
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import concat, col, udf, lag, date_add, explode, lit, unix_timestamp, regexp_extract, get_json_object
from pyspark.sql.functions import month, weekofyear, dayofmonth, year, hour, minute, second, to_timestamp
from pyspark.sql.types import *
from pyspark.sql.types import DateType
from pyspark.sql.types import DataType
from pyspark.sql.window import Window
from pyspark.sql import Row
from pyspark.ml.classification import *
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler,OneHotEncoder,VectorIndexer, PCA, RFormula
from pyspark.ml import Pipeline, PipelineModel
from delta import DeltaTable

# Data Cleansing

In [0]:
df = spark.read.table("fraud_detection_project.bronze_layer.customer_profiles")

# Standardize column names
def StandardizeNames(df):
    l = df.columns                                                  # (Regex Operator -> https://regex101.com/)
    cols = [re.sub(r'(?<!^)(?=[A-Z])', '_', c).lower() for c in l]  # Convert CamelCase to snake_case
    cols = [c.lstrip('_') for c in cols]  # Remove underscores in the beginning of column names
    return df.toDF(*cols)
df = StandardizeNames(df)

In [0]:
# Deleting duplicated data
df.dropDuplicates(['customer_id'])

# Deleting rows without some features
df = df.dropna(how='any', subset=['customer_id','file_path','ingest_datetime'])

In [0]:
# {"card_brand": "Mastercard", "card_category": "Black", "card_type": "D\u00e9bito", "security_code": "969", "issue_date": "2025-02-20", "expiration_date": "2028-09-30", "card_limit": 40172.24, "available_limit": 11102.13}

# Extracting directly the keys from card_details in the JSON
df = df.withColumn("card_brand",      get_json_object(col("card_details"), "$.card_brand"))
df = df.withColumn("card_category",   get_json_object(col("card_details"), "$.card_category"))
df = df.withColumn("card_type",       get_json_object(col("card_details"), "$.card_type"))
df = df.withColumn("security_code",   get_json_object(col("card_details"), "$.security_code"))
df = df.withColumn("issue_date",      get_json_object(col("card_details"), "$.issue_date"))
df = df.withColumn("expiration_date", get_json_object(col("card_details"), "$.expiration_date"))
df = df.withColumn("card_limit",      get_json_object(col("card_details"), "$.card_limit"))
df = df.withColumn("available_limit", get_json_object(col("card_details"), "$.available_limit"))

In [0]:
# {"customer_gender": "M", "customer_age": 84}

# Extracting directly the keys from client_details in the JSON
df = df.withColumn("customer_gender", get_json_object(col("client_details"), "$.customer_gender"))
df = df.withColumn("customer_age",    get_json_object(col("client_details"), "$.customer_age"))

In [0]:
# Deleting the column card_details
df = df.drop("card_details", "client_details")

In [0]:
df.createOrReplaceTempView("df1")

In [0]:
%sql
SELECT * FROM df1 LIMIT(1)

# Feature Engineering